### product_index.csv

    part_number,name,brand,category,url

    PS18241743,DISHWASHER TOUCH-UP PAINT,Midea,Dishwasher,https://www.partselect.com/...
    PS12345678,Door Latch Assembly,Whirlpool,Dishwasher,https://...

In [2]:
import os
import json
import csv

input_folder = r".\database\product_json_db"
output_file = "product_index.csv"

rows = []

for file in os.listdir(input_folder):
    if not file.endswith(".json"):
        continue
    with open(os.path.join(input_folder, file), "r", encoding="utf-8") as f:
        item = json.load(f)

        rows.append([
            item.get("partselect_number", ""),
            item.get("title", ""),
            item.get("brand", ""),
            item.get("category", ""),
            item.get("url", ""),
        ])

with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["part_number", "name", "brand", "category", "url"])
    writer.writerows(rows)


### repairs_index.csv

    filename,page_title,symptom,quick_nav_topics

    WILL_NOT_START - dishwasher.json,How to Fix a Dishwasher That Won't Start,WILL NOT START,"Door Latch|Timer|Selector Switch|Motor Start Relay|Thermal Fuse|Drive Motor"

In [13]:


input_folder = r"C:\Users\trrsh\Downloads\InstaLily AI\bot2\database\repairs_json_db"
output_file = "repairs_index2.csv"

rows = []

for file in os.listdir(input_folder):
    if not file.endswith(".json"):
        continue
    
    with open(os.path.join(input_folder, file), "r", encoding="utf-8") as f:
        item = json.load(f)

        symptom = item.get("symptom", "")
        page_title = item.get("page_title", "")
        model_type = item.get("data_modeltype", "Unknown")

        # Create filename in desired format
        #filename = f"{symptom} - {model_type.lower()}.json"
        filename = file

        # Build quick nav topics string
        quick_nav_topics = "|".join(
            [x["part_name"] for x in item.get("quick_navigation", []) 
             if x.get("part_name")]
        )

        rows.append([filename, page_title, symptom, quick_nav_topics])

# Write CSV
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "page_title", "symptom", "quick_nav_topics"])
    writer.writerows(rows)


### categories_mapping.csv
    category_file,brand,appliance_type,subcategories
    Admiral-Dishwasher.json,Admiral,Dishwasher,Door Latch|Spray Arm|Pump

In [4]:


input_folder = r".\database\categories_json_db"
output_csv = "categories_mapping.csv"

rows = []

for file in os.listdir(input_folder):
    if not file.endswith(".json"):
        continue

    filepath = os.path.join(input_folder, file)

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    brand = data.get("brand", "")
    appliance = data.get("model_type", "")

    # Extract names directly from popular_parts
    popular_parts = data.get("popular_parts", [])
    subcategories = [p.get("name", "") for p in popular_parts if p.get("name")]

    # Join with |
    subcat_str = "|".join(subcategories)

    rows.append([file, brand, appliance, subcat_str])

# Write CSV
with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "brand", "appliance_type", "subcategories"])
    writer.writerows(rows)

print("Done →", output_csv)


Done → categories_mapping.csv


### ptls_subcategory.csv

    ptls_file,parent_category,brand,appliance
    Amana-Refrigerator-Manuals-and.json,Refrigerator,Amana,Refrigerator

In [10]:
import os
import json
import csv

input_folder = r".\database\ptls_json_cleaned"
output_csv = "ptls_subcategory.csv"

rows = []

for file in os.listdir(input_folder):
    filepath = os.path.join(input_folder, file)

    if not os.path.isfile(filepath):
        continue  # skip folders

    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

        data = None
        try:
            # Try to load directly
            data = json.loads(content)
        except json.JSONDecodeError:
            # Try to extract JSON from messy content
            start = content.find("{")
            end = content.rfind("}") + 1
            if start != -1 and end != -1:
                try:
                    data = json.loads(content[start:end])
                except json.JSONDecodeError:
                    print(f"Skipping {file}: cannot parse JSON")
                    continue
            else:
                print(f"Skipping {file}: no JSON found")
                continue

    # Extract fields
    brand = data.get("brand", "")
    parent_category = data.get("model_type", "")
    parts_list = data.get("parts", [])
    appliances = [p.get("name", "") for p in parts_list if p.get("name")]
    appliance_str = "|".join(appliances)

    rows.append([file, parent_category, brand, appliance_str])

# Write CSV
with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["ptls_file", "parent_category", "brand", "appliance"])
    writer.writerows(rows)

print("Done →", output_csv)


Done → ptls_subcategory.csv


In [9]:
import os
import json

input_folder = r".\database\ptls_json_db"
output_folder = r".\database\ptls_json_cleaned"
os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    filepath = os.path.join(input_folder, file)
    
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Try to locate JSON block
    start = content.find("{")
    end = content.rfind("}") + 1
    
    if start == -1 or end == -1:
        print(f"Skipping {file}: no JSON found")
        continue

    json_str = content[start:end]
    
    try:
        data = json.loads(json_str)  # validate JSON
    except json.JSONDecodeError:
        print(f"Skipping {file}: invalid JSON after extraction")
        continue

    # Save clean JSON file
    out_file = os.path.join(output_folder, f"{os.path.splitext(file)[0]}.json")
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

print("All files converted to clean JSON →", output_folder)


Skipping Hotpoint-Dishwasher-Hotpoint-Dishwasher_4.json: invalid JSON after extraction
Skipping Whirlpool-Dishwasher-Whirlpool-Dishwasher_9.json: invalid JSON after extraction
All files converted to clean JSON → .\database\ptls_json_cleaned


### blogs_index.csv

In [2]:
import os
import json
import csv

input_folder = r".\database\blogs_json_db"
output_file = "blogs_index.csv"

rows = []

for file in os.listdir(input_folder):
    if not file.endswith(".json"):
        continue
    try:
        with open(os.path.join(input_folder, file), "r", encoding="utf-8") as f:
            item = json.load(f)
            
            # Extract parts mentioned safely
            parts = []
            for section in item.get("content_sections", []):
                for block in section.get("content", []):
                    for link in block.get("links") or []:
                        parts.append(link.get("text", ""))

            # Flatten headings
            headings = [sec.get("heading", "") for sec in item.get("content_sections", [])]

            # Optional: basic symptom extraction from headings
            symptoms = [h for h in headings if any(k in h.lower() for k in ["error", "won't start", "too hot", "leak"])]

            rows.append([
                file,  # <-- filename as first column
                item.get("url", ""),
                item.get("subtitle", ""),
                item.get("meta_description", ""),
                item.get("page_type", ""),
                "; ".join(symptoms),
                "; ".join(parts),
                "; ".join(headings)
            ])
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Write CSV
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "filename", "url", "subtitle", "meta_description", 
        "appliance_type", "symptoms", "parts_mentioned", "headings"
    ])
    writer.writerows(rows)

print(f"CSV written to {output_file} with {len(rows)} rows")


CSV written to blogs_index.csv with 38 rows


### all part, model csv

In [5]:
import os
import json
import csv

# Paths to your JSON databases
folders = {
    "products": r".\database\product_json_db",
    "ptls": r".\database\ptls_json_db",
    "categories": r".\database\categories_json_db"
}

output_file = "data.csv"

# Columns for the merged CSV
columns = [
    "json_path",
    "partselect_number",
    "manufacturer_part_number",
    "name",
    "brand",
    "category",
    "price",
    "original_price",
    "discount",
    "stock_status",
    "url",
    "rating",
    "review_count",
    "description",
    "fixes_symptoms",
    "images",
    "has_video",
    "cart_data",
    "installation_story"
]

rows = []

def process_folder(folder_name, folder_path):
    for file in os.listdir(folder_path):
        if not file.endswith(".json"):
            continue
        path = os.path.join(folder_path, file)
        try:
            with open(path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            items = data if isinstance(data, list) else [data]
            
            for data_item in items:
                row = {
                    "json_path": path,
                    "partselect_number": data_item.get("partselect_number") or "",
                    "manufacturer_part_number": data_item.get("manufacturer_part_number") or "",
                    "name": data_item.get("name") or data_item.get("title") or "",
                    "brand": data_item.get("brand") or "",
                    "category": data_item.get("category") or "",
                    "price": data_item.get("price") or "",
                    "original_price": data_item.get("original_price") or "",
                    "discount": data_item.get("discount") or "",
                    "stock_status": data_item.get("stock_status") or data_item.get("availability") or "",
                    "url": data_item.get("url") or "",
                    "rating": data_item.get("rating") or "",
                    "review_count": data_item.get("review_count") or "",
                    "description": data_item.get("description") or "",
                    "fixes_symptoms": "; ".join(data_item.get("fixes_symptoms", [])) if data_item.get("fixes_symptoms") else "",
                    "images": "; ".join(data_item.get("images", [])) if data_item.get("images") else "",
                    "has_video": data_item.get("has_video") or False,
                    "cart_data": data_item.get("cart_data") or "",
                    "installation_story": data_item.get("installation_story") or ""
                }
                
                rows.append(row)
        
        except Exception as e:
            print(f"❌ Error processing {path}: {e}")

# Process each folder
for name, path in folders.items():
    print(f"Processing {name} JSONs in {path}")
    if os.path.exists(path):
        process_folder(name, path)
    else:
        print(f"⚠️ Folder not found: {path}")

# Write merged CSV
with open(output_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)

print(f"\n✅ Merged CSV saved to {output_file}, total rows: {len(rows)}")


Processing products JSONs in .\database\product_json_db
Processing ptls JSONs in .\database\ptls_json_db
❌ Error processing .\database\ptls_json_db\Hotpoint-Dishwasher-Hotpoint-Dishwasher_4.json: Extra data: line 313 column 6 (char 13163)
❌ Error processing .\database\ptls_json_db\Whirlpool-Dishwasher-Whirlpool-Dishwasher_9.json: Extra data: line 373 column 2 (char 13658)
Processing categories JSONs in .\database\categories_json_db

✅ Merged CSV saved to data.csv, total rows: 2601


In [6]:
import os
import json
import csv

folders = {
    "products": r".\database\product_json_db",
    "ptls": r".\database\ptls_json_db",
    "categories": r".\database\categories_json_db"
}

output_csv = r".\all_parts_combined.csv"

fields = [
    "source_folder", "file_name", "part_name", "partselect_number", "manufacturer_part_number",
    "price", "original_price", "discount", "stock_status", "rating",
    "review_count", "description", "fixes_symptoms", "installation_author",
    "installation_title", "installation_description", "part_url", "image_url"
]

rows = []

for source, folder in folders.items():
    for file in os.listdir(folder):
        if not file.endswith(".json"):
            continue
        filepath = os.path.join(folder, file)

        with open(filepath, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                print(f"Skipping invalid JSON: {file} in {source}")
                continue

        # Collect lists of parts depending on structure
        part_lists = []

        # ptls: usually "parts"
        if "parts" in data:
            part_lists.append(data["parts"])

        # some product JSON may have "popular_parts" or "parts"
        if "popular_parts" in data:
            part_lists.append(data["popular_parts"])
        if "parts" in data:
            part_lists.append(data["parts"])

        # categories: may have parts under different keys, fallback example
        if "category_parts" in data:
            part_lists.append(data["category_parts"])

        for part_list in part_lists:
            for part in part_list:
                row = {
                    "source_folder": source,
                    "file_name": file,
                    "part_name": part.get("name"),
                    "partselect_number": part.get("partselect_number"),
                    "manufacturer_part_number": part.get("manufacturer_part_number"),
                    "price": part.get("price"),
                    "original_price": part.get("original_price"),
                    "discount": part.get("discount"),
                    "stock_status": part.get("stock_status"),
                    "rating": part.get("rating"),
                    "review_count": part.get("review_count"),
                    "description": part.get("description"),
                    "fixes_symptoms": ", ".join(part.get("fixes_symptoms", [])) if part.get("fixes_symptoms") else "",
                    "installation_author": part.get("installation_story", {}).get("author") if part.get("installation_story") else "",
                    "installation_title": part.get("installation_story", {}).get("title") if part.get("installation_story") else "",
                    "installation_description": part.get("installation_story", {}).get("description") if part.get("installation_story") else "",
                    "part_url": part.get("url"),
                    "image_url": part.get("image", {}).get("url") if part.get("image") else part.get("brand_image", {}).get("url") if part.get("brand_image") else ""
                }
                rows.append(row)

# Write combined CSV
with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)

print(f"Done! Saved {len(rows)} items from all folders to {output_csv}")


Skipping invalid JSON: Hotpoint-Dishwasher-Hotpoint-Dishwasher_4.json in ptls
Skipping invalid JSON: Whirlpool-Dishwasher-Whirlpool-Dishwasher_9.json in ptls
Done! Saved 14056 items from all folders to .\all_parts_combined.csv


In [7]:
## random sample for evaluation set

import pandas as pd


In [9]:
dir = r"C:\Users\trrsh\Downloads\InstaLily AI\bot2\database"

df1 = pd.read_csv(dir + r'\product_index.csv')
df2 = pd.read_csv(dir + r'\repairs_index.csv')
df3 = pd.read_csv(dir + r'\blogs_index.csv')

In [10]:
prod_sample = df1.sample(100)
repair_sample = df2.sample(35)
blogs_sample = df3.sample(20)

In [11]:
prod_sample.to_csv('prod_sample.csv', index = False)
repair_sample.to_csv('repair_sample.csv', index = False)
blogs_sample.to_csv('blogs_sample.csv', index = False)